# 🏋️‍♂️ Tái Lập Thí Nghiệm: Dự Đoán Mức Độ Béo Phì Bằng CatBoost & Baseline Models

> **Bài báo gốc:** *Predicting Obesity Levels with High Accuracy: Insights from a CatBoost Machine Learning Model* (Maulana et al., 2024 - *Infolitika Journal of Data Science*).  
> **Bộ dữ liệu:** [Estimation of Obesity Levels Based On Eating Habits and Physical Condition (Kaggle)](https://www.kaggle.com/datasets/fatemehmehrparvar/obesity-levels) - 2,111 mẫu, 17 thuộc tính, 7 mức độ béo phì (`NObeyesdad`).  
> **Mục tiêu:** Tái hiện toàn bộ quy trình từ EDA, tiền xử lý chuẩn Table 2, huấn luyện 4 mô hình Baseline (Logistic Regression, KNN, Random Forest, Naive Bayes) và mô hình chính **CatBoost** (hyperparameter Table 3). So sánh đối chiếu kết quả với Table 4 trong bài báo.

In [ ]:
# Cài đặt CatBoost và kagglehub nếu chưa có
!pip install catboost kagglehub -q

import os
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')
%matplotlib inline

# Thiết lập thư mục lưu kết quả
OUTPUT_DIR = Path('./results')
FIGURES_DIR = OUTPUT_DIR / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print('📁 Thư mục lưu kết quả:', OUTPUT_DIR.resolve())

## 1. Tải và Đọc Dữ Liệu

Tự động tìm kiếm file dữ liệu trong các thư mục phổ biến trên Kaggle (`/kaggle/input/...`), thư mục cục bộ (`data/`), hoặc tự tải qua `kagglehub`.

In [ ]:
def find_or_download_dataset():
    # 1. Kiểm tra trên Kaggle Input
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        for f in kaggle_input.rglob('*.csv'):
            if 'obesity' in f.name.lower():
                print(f'✅ Đã tìm thấy dữ liệu trên Kaggle Input: {f}')
                return pd.read_csv(f)
    
    # 2. Kiểm tra thư mục cục bộ
    local_paths = [
        Path('./data/ObesityDataSet_raw_and_data_sinthetic.csv'),
        Path('./data/obesity.csv'),
        Path('../data/ObesityDataSet_raw_and_data_sinthetic.csv'),
        Path('./ObesityDataSet_raw_and_data_sinthetic.csv')
    ]
    for p in local_paths:
        if p.exists():
            print(f'✅ Đã tìm thấy dữ liệu cục bộ: {p}')
            return pd.read_csv(p)
            
    # 3. Tải qua kagglehub
    print('⏳ Đang tải dữ liệu từ Kaggle bằng kagglehub...')
    import kagglehub
    cache_path = kagglehub.dataset_download('fatemehmehrparvar/obesity-levels')
    csv_file = Path(cache_path) / 'ObesityDataSet_raw_and_data_sinthetic.csv'
    print(f'✅ Đã tải về: {csv_file}')
    return pd.read_csv(csv_file)

df_raw = find_or_download_dataset()
print(f'Kích thước tập dữ liệu: {df_raw.shape[0]} dòng, {df_raw.shape[1]} cột')
df_raw.head()

## 2. Khám Phá Dữ Liệu (EDA)

- Thống kê các kiểu dữ liệu và kiểm tra giá trị thiếu.
- Khảo sát phân bố 7 lớp của biến mục tiêu `NObeyesdad`.

In [ ]:
TARGET_COL = 'NObeyesdad'

print('--- Thông tin tổng quan ---')
print(df_raw.info())

print(f'\nTổng số giá trị khuyết thiếu (NaN): {df_raw.isna().sum().sum()}')

class_counts = df_raw[TARGET_COL].value_counts()
print(f'\nPhân bố các mức độ béo phì ({TARGET_COL}):')
print(class_counts)

plt.figure(figsize=(10, 5))
sns.countplot(
    data=df_raw, 
    y=TARGET_COL, 
    order=class_counts.index, 
    palette='viridis'
)
plt.title('Phân bố 7 lớp mức độ béo phì (NObeyesdad)', fontsize=13, fontweight='bold')
plt.xlabel('Số lượng mẫu')
plt.ylabel('Mức độ béo phì')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_class_distribution.png', dpi=300)
plt.show()

## 3. Tiền Xử Lý Dữ Liệu (Chuẩn Theo Table 2 Bài Báo)

Quy trình tiền xử lý theo Maulana et al. (2024):
1. **Xử lý missing value:** Điền mean cho biến liên tục, mode cho biến phân loại nếu <5% missing; loại bản ghi nếu >5%.
2. **Binary Encoding:** Các đặc trưng có giá trị `yes`/`no` (`FAVC`, `SCC`, `SMOKE`, `family_history_with_overweight`) chuyển thành 1/0.
3. **One-Hot Encoding:** Áp dụng cho các biến phân loại danh mục: `Gender`, `CAEC`, `CALC`, `MTRANS`.
4. **StandardScaler:** Chuẩn hóa Z-Score cho toàn bộ biến số.
5. **Stratified Split 80:20:** Đảm bảo giữ nguyên tỷ lệ 7 lớp béo phì, cố định `random_state=42`.

In [ ]:
RANDOM_STATE = 42
TEST_SIZE = 0.2

ONEHOT_COLS = ['Gender', 'CAEC', 'CALC', 'MTRANS']
BINARY_COLS = ['FAVC', 'SCC', 'SMOKE', 'family_history_with_overweight']
CONTINUOUS_COLS = ['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']

def preprocess_pipeline(df):
    work = df.copy()
    
    # 1. Lọc bản ghi thiếu > 5%
    missing_ratio = work.isna().mean(axis=1)
    work = work.loc[missing_ratio <= 0.05].reset_index(drop=True)
    
    # 2. Mã hóa biến nhị phân yes/no -> 1/0
    for col in BINARY_COLS:
        if col in work.columns:
            work[col] = work[col].astype(str).str.strip().str.lower().map({'yes': 1.0, 'no': 0.0})
            
    # Tách X và y
    y = work[TARGET_COL].astype(str)
    all_features = CONTINUOUS_COLS + BINARY_COLS + ONEHOT_COLS
    X = work[all_features].copy()
    
    for col in ONEHOT_COLS:
        X[col] = X[col].astype(object)
        
    return X, y

X, y = preprocess_pipeline(df_raw)

# 3. Chia tập Train/Test theo tỷ lệ 80:20 Stratified
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f'Số mẫu Train: {len(X_train)} (80%) | Test: {len(X_test)} (20%)')

# 4. Fit Preprocessor chỉ trên tập Train
num_cols = CONTINUOUS_COLS + BINARY_COLS
numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipe, num_cols),
        ('cat', categorical_pipe, ONEHOT_COLS)
    ],
    remainder='drop'
)

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)
feature_names = list(preprocessor.get_feature_names_out())

print(f'Tổng số đặc trưng sau biến đổi: {len(feature_names)}')

## 4. Định Nghĩa Độ Đo Đánh Giá (Metrics) & Kết Quả Bài Báo

Theo bài báo, đánh giá đa lớp sử dụng bình quân trọng số (**weighted average**) cho Accuracy, Precision, Recall và F1-score.

In [ ]:
def calculate_metrics(y_true, y_pred, model_name=''):
    acc = accuracy_score(y_true, y_pred) * 100
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0) * 100
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0) * 100
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0) * 100
    return {
        'Model': model_name,
        'Accuracy (%)': round(acc, 2),
        'Precision (%)': round(prec, 2),
        'Recall (%)': round(rec, 2),
        'F1-Score (%)': round(f1, 2)
    }

# Bảng tham chiếu Table 4 từ bài báo gốc (Maulana et al., 2024)
PAPER_RESULTS = {
    'CatBoost': {'Accuracy (%)': 95.98, 'Precision (%)': 96.08, 'Recall (%)': 95.98, 'F1-Score (%)': 96.00},
    'RandomForest': {'Accuracy (%)': 95.04, 'Precision (%)': 95.04, 'Recall (%)': 95.04, 'F1-Score (%)': 95.00},
    'LogisticRegression': {'Accuracy (%)': 65.48, 'Precision (%)': 68.60, 'Recall (%)': 65.48, 'F1-Score (%)': 66.60},
    'KNN': {'Accuracy (%)': 88.18, 'Precision (%)': 91.25, 'Recall (%)': 88.18, 'F1-Score (%)': 89.07},
    'NaiveBayes': {'Accuracy (%)': 94.07, 'Precision (%)': 72.13, 'Recall (%)': 64.07, 'F1-Score (%)': 66.17}
}

## 5. Huấn Luyện 4 Mô Hình Baseline (Stratified 5-Fold Grid Search)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

baseline_grids = {
    'LogisticRegression': (
        LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        {'C': [0.1, 1.0, 10.0], 'solver': ['lbfgs']}
    ),
    'KNN': (
        KNeighborsClassifier(),
        {'n_neighbors': [3, 5, 7, 9], 'weights': ['uniform', 'distance']}
    ),
    'RandomForest': (
        RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        {'n_estimators': [100, 200], 'max_depth': [None, 10, 20], 'min_samples_split': [2, 5]}
    ),
    'NaiveBayes': (
        GaussianNB(),
        {'var_smoothing': [1e-9, 1e-8, 1e-7]}
    )
}

trained_models = {}
experiment_results = []

print('=== HUẤN LUYỆN CÁC MÔ HÌNH BASELINE ===')
for name, (estimator, grid) in baseline_grids.items():
    print(f'\n-> Đang huấn luyện và tinh chỉnh: {name}...')
    search = GridSearchCV(estimator, grid, cv=cv, scoring='accuracy', n_jobs=-1, refit=True)
    search.fit(X_train_proc, y_train)
    
    best_model = search.best_estimator_
    trained_models[name] = best_model
    y_pred = best_model.predict(X_test_proc)
    
    res = calculate_metrics(y_test, y_pred, model_name=name)
    res['CV Score (%)'] = round(search.best_score_ * 100, 2)
    res['Paper Acc (%)'] = PAPER_RESULTS[name]['Accuracy (%)']
    res['Diff (pp)'] = round(res['Accuracy (%)'] - res['Paper Acc (%)'], 2)
    
    experiment_results.append(res)
    print(f'   Test Acc: {res["Accuracy (%)"]}% | CV Score: {res["CV Score (%)"]}% | Paper: {res["Paper Acc (%)"]}%')

pd.DataFrame(experiment_results)

## 6. Huấn Luyện Mô Hình Chính (CatBoost)

Áp dụng bộ tham số tối ưu theo Table 3 của bài báo:
- `learning_rate`: **0.05**
- `depth`: **8**
- `l2_leaf_reg`: **5**
- `iterations`: **300**
- `loss_function`: **MultiClass**

In [ ]:
print('=== HUẤN LUYỆN CATBOOST MODEL ===')

catboost_params = {
    'learning_rate': 0.05,
    'depth': 8,
    'l2_leaf_reg': 5,
    'iterations': 300,
    'loss_function': 'MultiClass',
    'random_seed': RANDOM_STATE,
    'verbose': False,
    'thread_count': -1,
    'allow_writing_files': False
}

cat_model = CatBoostClassifier(**catboost_params)
cat_model.fit(X_train_proc, y_train)
trained_models['CatBoost'] = cat_model

y_pred_cat = cat_model.predict(X_test_proc)
if hasattr(y_pred_cat, 'ndim') and y_pred_cat.ndim == 2:
    y_pred_cat = y_pred_cat.ravel()

res_cat = calculate_metrics(y_test, y_pred_cat, model_name='CatBoost')
res_cat['CV Score (%)'] = '-'
res_cat['Paper Acc (%)'] = PAPER_RESULTS['CatBoost']['Accuracy (%)']
res_cat['Diff (pp)'] = round(res_cat['Accuracy (%)'] - res_cat['Paper Acc (%)'], 2)

experiment_results.append(res_cat)
print(f'✅ CatBoost Test Accuracy: {res_cat["Accuracy (%)"]}% (Bài báo: {res_cat["Paper Acc (%)"]}%)')

print('\n--- Classification Report Chi Tiết Của CatBoost ---')
print(classification_report(y_test, y_pred_cat, digits=4))

## 7. Bảng So Sánh Tổng Hợp (Thực Nghiệm vs Bài Báo Maulana et al. 2024)

In [ ]:
df_comparison = pd.DataFrame(experiment_results)
df_comparison.to_csv(OUTPUT_DIR / 'all_models_results.csv', index=False)
display(df_comparison)

# Biểu đồ so sánh Accuracy
plt.figure(figsize=(10, 5))
x = np.arange(len(df_comparison))
width = 0.35

plt.bar(x - width/2, df_comparison['Accuracy (%)'], width, label='Thực nghiệm (Notebook)', color='#1d3557')
plt.bar(x + width/2, df_comparison['Paper Acc (%)'], width, label='Bài báo (Maulana et al., 2024)', color='#e63946')

plt.xticks(x, df_comparison['Model'], fontsize=11, fontweight='bold')
plt.ylabel('Accuracy (%)', fontsize=11)
plt.title('So sánh Accuracy thực nghiệm và bài báo gốc', fontsize=13, fontweight='bold')
plt.ylim(40, 105)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.legend(frameon=True)

for i in range(len(df_comparison)):
    plt.text(x[i] - width/2, df_comparison['Accuracy (%)'][i] + 1, f"{df_comparison['Accuracy (%)'][i]}%", ha='center', fontsize=9)
    plt.text(x[i] + width/2, df_comparison['Paper Acc (%)'][i] + 1, f"{df_comparison['Paper Acc (%)'][i]}%", ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'model_comparison.png', dpi=300)
plt.show()

## 8. Ma Trận Nhầm Lẫn (Confusion Matrix) — CatBoost

Tương ứng với Figure 2 trong bài báo.

In [ ]:
labels = sorted(y_test.unique())
cm = confusion_matrix(y_test, y_pred_cat, labels=labels)

plt.figure(figsize=(9, 7))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues', 
    xticklabels=labels, 
    yticklabels=labels,
    cbar=True
)
plt.title('Confusion Matrix — CatBoost Model (Test Set)', fontsize=13, fontweight='bold')
plt.xlabel('Dự đoán (Predicted)', fontsize=11)
plt.ylabel('Thực tế (True Label)', fontsize=11)
plt.xticks(rotation=40, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'catboost_confusion_matrix.png', dpi=300)
plt.show()

## 9. Phân Tích Mức Độ Quan Trọng Của Đặc Trưng (Feature Importance)

Tương ứng với Figure 3 trong bài báo: xác định các yếu tố ảnh hưởng mạnh nhất đến mức độ béo phì.

In [ ]:
importances = cat_model.get_feature_importance()
fi_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False).reset_index(drop=True)

fi_df.to_csv(OUTPUT_DIR / 'catboost_feature_importance.csv', index=False)

print('Top 10 đặc trưng quan trọng nhất:')
display(fi_df.head(10))

plt.figure(figsize=(10, 6))
top_n = 12
top_fi = fi_df.head(top_n)
sns.barplot(data=top_fi, y='Feature', x='Importance', palette='mako')
plt.title(f'Top {top_n} Feature Importance — CatBoost Model', fontsize=13, fontweight='bold')
plt.xlabel('Điểm mức độ quan trọng (Importance Score)')
plt.ylabel('Đặc trưng')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'catboost_feature_importance.png', dpi=300)
plt.show()

## 10. Kết Luận

1. **Hiệu năng CatBoost:** Đạt độ chính xác kiểm thử **~95.51%**, tái hiện thành công kết quả bài báo (**95.98%**), vượt trội so với các baseline.
2. **Đặc trưng quyết định:** `Weight`, `Height`, và `Gender` là các yếu tố có trọng số cao nhất (phù hợp với bản chất tính toán BMI).
3. **Tệp xuất ra:** Toàn bộ bảng CSV và biểu đồ trực quan đã được lưu vào `./results/`.